# Assignment 1, Task 1: MLPs and CNNs on Fashion-MNIST and CIFAR-10
Run top to bottom on a **GPU runtime** (Runtime > Change runtime type > GPU). Selection always uses the validation set; the test sets are only used in the final section.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)   # should say cuda; on cpu the 15-epoch runs are very slow
torch.manual_seed(42)

## 1. Data (90/10 train/validation split)

In [ ]:
batch_size = 128

def make_loaders(dataset_cls, mean, std):
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    full = dataset_cls(root='./data', train=True, download=True, transform=tf)
    test = dataset_cls(root='./data', train=False, download=True, transform=tf)
    n_val = int(0.1 * len(full))
    tr, va = random_split(full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(42))
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size),
            DataLoader(test, batch_size=batch_size), full)

fm_train_loader, fm_val_loader, fm_test_loader, fm_full = make_loaders(datasets.FashionMNIST, (0.2860,), (0.3530,))
c_train_loader, c_val_loader, c_test_loader, c_full = make_loaders(
    datasets.CIFAR10, (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
print(len(fm_train_loader.dataset), len(fm_val_loader.dataset), len(fm_test_loader.dataset))
print(len(c_train_loader.dataset), len(c_val_loader.dataset), len(c_test_loader.dataset))

In [ ]:
imgs, labels = next(iter(fm_train_loader))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, lbl in zip(axes.flat, imgs, labels):
    ax.imshow(img.squeeze(), cmap='gray'); ax.set_title(fm_full.classes[lbl], fontsize=9); ax.axis('off')
plt.show()

## 2. Models

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=28*28, hidden_sizes=(256, 128), n_classes=10,
                 activation=nn.ReLU, dropout=0.0):
        super().__init__()
        layers = [nn.Flatten()]
        prev = in_features
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), activation(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))   # raw logits; CrossEntropyLoss applies softmax
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, in_channels=1, img_size=28, channels=(32, 64), kernel_size=3,
                 fc_size=128, n_classes=10, activation=nn.ReLU, dropout=0.0):
        super().__init__()
        layers, prev = [], in_channels
        for c in channels:
            layers += [nn.Conv2d(prev, c, kernel_size, padding=kernel_size // 2), activation(), nn.MaxPool2d(2)]
            prev = c
        self.features = nn.Sequential(*layers)
        final_size = img_size // (2 ** len(channels))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(prev * final_size * final_size, fc_size),
                                        activation(), nn.Dropout(dropout), nn.Linear(fc_size, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def init_weights(model, scheme):
    if scheme == 'default':
        return
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            if scheme == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif scheme == 'xavier':
                nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

## 3. Training and evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def train(model, optimizer, train_loader, val_loader, epochs=10, verbose=True):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
        val_loss, val_acc = evaluate(model, val_loader)
        for k, v in zip(history, [total_loss / n, correct / n, val_loss, val_acc]):
            history[k].append(v)
        if verbose:
            print(f"Epoch {epoch+1:2d}/{epochs} | train loss {total_loss/n:.4f} acc {correct/n:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
    return history

## 4. Experiment runner
`run` builds a model, trains it, and logs one row in `results`. `ds` picks Fashion-MNIST (`'fm'`) or CIFAR-10 (`'cifar'`), and the right input size is filled in automatically.

In [ ]:
results = []
hists = {}

data = {
    'fm':    dict(train=fm_train_loader, val=fm_val_loader, test=fm_test_loader,
                  mlp=dict(in_features=28*28), cnn=dict(in_channels=1, img_size=28)),
    'cifar': dict(train=c_train_loader, val=c_val_loader, test=c_test_loader,
                  mlp=dict(in_features=3*32*32), cnn=dict(in_channels=3, img_size=32)),
}

def run(group, net, name, model_kwargs=None, opt_name='adam', lr=1e-3, wd=0.0, init='default',
        epochs=5, seed=42, ds='fm', test=False, log=True):
    torch.manual_seed(seed)
    d = data[ds]
    model = (MLP if net == 'mlp' else CNN)(**{**d[net], **(model_kwargs or {})}).to(device)
    init_weights(model, init)
    if opt_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    elif opt_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)      # L2 penalty
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)     # decoupled decay
    h = train(model, optimizer, d['train'], d['val'], epochs=epochs, verbose=False)
    test_acc = evaluate(model, d['test'])[1] if test else None   # final-epoch weights
    results.append(dict(group=group, ds=ds, net=net, name=name, seed=seed, epochs=epochs,
                        best_val_acc=max(h['val_acc']), final_train_acc=h['train_acc'][-1],
                        test_acc=test_acc, n_params=sum(p.numel() for p in model.parameters()),
                        cfg=dict(model_kwargs=model_kwargs or {}, opt_name=opt_name, lr=lr, wd=wd, init=init)))
    if log:
        print(f"[{ds}/{net}] {group:8s} {name:45s} seed {seed} ep {epochs:2d} val {max(h['val_acc']):.4f}")
    return dict(val=max(h['val_acc']), hist=h, model=model)

## 5. Baselines (verbose, with learning curves)

In [ ]:
base = {}
for net in ['mlp', 'cnn']:
    torch.manual_seed(42)
    m = (MLP if net == 'mlp' else CNN)(**data['fm'][net]).to(device)
    base[net] = train(m, optim.Adam(m.parameters(), lr=1e-3), fm_train_loader, fm_val_loader, epochs=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for net, h in base.items():
    axes[0].plot(h['train_loss'], label=f'{net} train'); axes[0].plot(h['val_loss'], '--', label=f'{net} val')
    axes[1].plot(h['train_acc'], label=f'{net} train'); axes[1].plot(h['val_acc'], '--', label=f'{net} val')
axes[0].set_title('Loss'); axes[1].set_title('Accuracy')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend()
plt.show()

## 6. Stage 1: screening, one factor at a time (5 epochs, Fashion-MNIST)
Coupled pairs (optimizer x lr, activation x init) are crossed. Adam and AdamW are identical at wd=0, so AdamW is only compared in the weight-decay experiments.

In [ ]:
for net in ['mlp', 'cnn']:
    run('baseline', net, 'default')

# optimizer x learning rate (lr grids shifted per optimizer on purpose)
lr_grid = {'sgd': [1e-3, 1e-2, 1e-1, 2e-1, 3e-1], 'adam': [1e-4, 1e-3, 1e-2]}
for net in ['mlp', 'cnn']:
    for opt_name, lrs in lr_grid.items():
        for lr in lrs:
            run('opt x lr', net, f'{opt_name} lr={lr}', opt_name=opt_name, lr=lr)

In [ ]:
# activation x initialization
for net in ['mlp', 'cnn']:
    for act in [nn.ReLU, nn.LeakyReLU, nn.GELU, nn.Tanh]:
        for init in ['default', 'kaiming', 'xavier']:
            run('act x init', net, f'{act.__name__} + {init}', dict(activation=act), init=init)

In [ ]:
# regularization
for net in ['mlp', 'cnn']:
    for p in [0.2, 0.5]:
        run('dropout', net, f'dropout={p}', dict(dropout=p))
    for wd in [1e-5, 1e-4, 1e-3]:
        run('wd', net, f'adam wd={wd}', wd=wd)

# architecture
for hs in [(128,), (256, 128, 64), (512, 256, 128, 64)]:
    run('depth', 'mlp', f'hidden={hs}', dict(hidden_sizes=hs))
for hs in [(64, 64), (512, 512), (1024, 1024)]:
    run('width', 'mlp', f'hidden={hs}', dict(hidden_sizes=hs))
for ch in [(32,), (32, 64, 128)]:
    run('depth', 'cnn', f'channels={ch}', dict(channels=ch))
for k in [5, 7]:
    run('kernel', 'cnn', f'kernel={k}', dict(kernel_size=k))
for ch in [(16, 32), (64, 128)]:
    run('filters', 'cnn', f'channels={ch}', dict(channels=ch))
for fc in [64, 512]:
    run('fc width', 'cnn', f'fc_size={fc}', dict(fc_size=fc))

In [ ]:
# seed noise of the baseline, and longer (15 epoch) regularization tests
for net in ['mlp', 'cnn']:
    for seed in [0, 1, 2]:
        run('noise', net, 'baseline', seed=seed, log=False)
    run('long', net, 'baseline 15ep', epochs=15)
    for p in [0.2, 0.5]:
        run('long', net, f'dropout={p} 15ep', dict(dropout=p), epochs=15)
    for wd in [1e-4, 1e-3]:
        run('long', net, f'adam wd={wd} 15ep', wd=wd, epochs=15)
    for wd in [1e-2, 1e-1]:
        run('long', net, f'adamw wd={wd} 15ep', opt_name='adamw', wd=wd, epochs=15)

df = pd.DataFrame(results)
print(df[df.group == 'noise'].groupby('net')['best_val_acc'].agg(['mean', 'std']))

In [ ]:
df = pd.DataFrame(results)
df['gap'] = df.final_train_acc - df.best_val_acc    # rough overfitting indicator
s1 = df[df.group != 'stage2']
for net in ['mlp', 'cnn']:
    print(f"\n=== {net.upper()} stage 1 ===")
    print(s1[(s1.net == net) & (s1.ds == 'fm')].sort_values('best_val_acc', ascending=False)
            [['group', 'name', 'epochs', 'best_val_acc', 'gap', 'n_params']].to_string(index=False))

### Stage 1 decision log (write this after reading the tables above)
- Noise level (std over 3 seeds):
- Factors that moved accuracy by more than ~2x noise:
- Factors that stay fixed from now on:
- Values carried into Stage 2:
- Caveats: 5-epoch runs favour fast convergence; most runs are single-seed; validation set of 6,000 images limits resolution to about 0.4 points.

## 7. Stage 2: factorial grid over the factors that matter (15 epochs)
This grid is the standard recipe. If your Stage 1 log points elsewhere, edit the value lists below. Because second-best values are included, it checks whether locking factors too early would have missed a better combination.

In [ ]:
for act in [nn.ReLU, nn.GELU]:
    for opt_name, wd in [('adam', 0.0), ('adamw', 0.01)]:
        for hs in [(256, 128), (512, 512)]:
            run('stage2', 'mlp', f'{act.__name__} | {opt_name} wd={wd} | {hs}',
                dict(activation=act, hidden_sizes=hs), opt_name=opt_name, wd=wd, epochs=15)

for act in [nn.ReLU, nn.LeakyReLU, nn.GELU]:
    for p in [0.0, 0.2]:
        for ch in [(32, 64), (64, 128)]:
            run('stage2', 'cnn', f'{act.__name__} | dropout={p} | {ch}',
                dict(activation=act, dropout=p, channels=ch), epochs=15)

df = pd.DataFrame(results)
s2 = df[df.group == 'stage2'].copy()
s2['gap'] = s2.final_train_acc - s2.best_val_acc
for net in ['mlp', 'cnn']:
    print(f"\n=== {net.upper()} stage 2 ===")
    print(s2[s2.net == net].sort_values('best_val_acc', ascending=False)[['name', 'best_val_acc', 'gap', 'n_params']].to_string(index=False))

## 8. Stage 3: confirm the top 3 per network with 3 seeds
The top 3 Stage 2 configurations (by validation accuracy) are picked automatically and re-run with 3 seeds. Rank by the mean, not a single run.

In [ ]:
chosen = {}
for net in ['mlp', 'cnn']:
    top = s2[s2.net == net].sort_values('best_val_acc', ascending=False).head(3)
    for i, (_, row) in enumerate(top.iterrows()):
        chosen[f'{net}_top{i+1}'] = dict(net=net, cfg=row.cfg, desc=row['name'])

for name, c in chosen.items():
    print(name, '=', c['desc'])
    for seed in [0, 1, 2]:
        run('stage3', c['net'], name, c['cfg']['model_kwargs'], c['cfg']['opt_name'],
            lr=c['cfg']['lr'], wd=c['cfg']['wd'], epochs=15, seed=seed, log=False)

df = pd.DataFrame(results)
print(df[df.group == 'stage3'].groupby(['net', 'name'])['best_val_acc'].agg(['mean', 'std']).round(4))

## 9. Final evaluation and CIFAR-10 transfer
The only place the test sets are used. Each chosen config and the baseline are trained with identical settings on both datasets (seed 42, 15 epochs).

In [ ]:
final_cfgs = {'mlp_baseline': dict(net='mlp', cfg=dict(model_kwargs={}, opt_name='adam', lr=1e-3, wd=0.0)),
              'cnn_baseline': dict(net='cnn', cfg=dict(model_kwargs={}, opt_name='adam', lr=1e-3, wd=0.0)),
              **chosen}

for name, c in final_cfgs.items():
    for ds in ['fm', 'cifar']:
        out = run('final', c['net'], name, c['cfg']['model_kwargs'], c['cfg']['opt_name'],
                  lr=c['cfg']['lr'], wd=c['cfg']['wd'], epochs=15, ds=ds, test=True, log=False)
        hists[(name, ds)] = out['hist']

df = pd.DataFrame(results)
fin = df[df.group == 'final'].copy()
fin['gap'] = fin.final_train_acc - fin.best_val_acc
print(fin.pivot_table(index=['net', 'name'], columns='ds', values=['best_val_acc', 'test_acc', 'gap']).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ds in zip(axes, ['fm', 'cifar']):
    for (name, d), h in hists.items():
        if d == ds: ax.plot(h['val_acc'], label=name)
    ax.set_title(f'Validation accuracy ({ds})'); ax.set_xlabel('Epoch'); ax.legend(fontsize=7)
plt.show()

### Discussion prompts (CIFAR-10 vs Fashion-MNIST)
- Did the ranking of configurations change between datasets?
- Compare the train-validation gap per dataset and network type: where does overfitting appear?
- Why would MLPs lose more on RGB natural images than CNNs (translation structure, local features, parameter sharing)?

## 10. Extra: Optuna search (same 15-epoch budget)

In [ ]:
!pip install -q optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
act_map = {'ReLU': nn.ReLU, 'LeakyReLU': nn.LeakyReLU, 'GELU': nn.GELU, 'Tanh': nn.Tanh}

def objective_mlp(trial):
    width = trial.suggest_categorical('width', [128, 256, 512])
    kw = dict(activation=act_map[trial.suggest_categorical('act', list(act_map))],
              hidden_sizes=(width,) * trial.suggest_int('n_layers', 1, 3),
              dropout=trial.suggest_float('dropout', 0.0, 0.3))
    return run('optuna', 'mlp', f'trial {trial.number}', kw, trial.suggest_categorical('opt', ['adam', 'adamw']),
               lr=trial.suggest_float('lr', 1e-4, 1e-2, log=True), wd=trial.suggest_float('wd', 1e-5, 1e-1, log=True),
               epochs=15, log=False)['val']

def objective_cnn(trial):
    ch = {'32-64': (32, 64), '64-128': (64, 128), '32-64-128': (32, 64, 128)}[
        trial.suggest_categorical('channels', ['32-64', '64-128', '32-64-128'])]
    kw = dict(activation=act_map[trial.suggest_categorical('act', list(act_map))], channels=ch,
              kernel_size=trial.suggest_categorical('kernel', [3, 5]), fc_size=trial.suggest_categorical('fc', [64, 128, 256]),
              dropout=trial.suggest_float('dropout', 0.0, 0.3))
    return run('optuna', 'cnn', f'trial {trial.number}', kw, trial.suggest_categorical('opt', ['adam', 'adamw']),
               lr=trial.suggest_float('lr', 1e-4, 1e-2, log=True), wd=trial.suggest_float('wd', 1e-5, 1e-1, log=True),
               epochs=15, log=False)['val']

studies = {}
for net, obj in [('mlp', objective_mlp), ('cnn', objective_cnn)]:
    studies[net] = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    studies[net].optimize(obj, n_trials=30)
    print(net, 'best val acc:', round(studies[net].best_value, 4), studies[net].best_params)
    print('importance:', optuna.importance.get_param_importances(studies[net]))

Compare each Optuna best value with the Stage 3 manual mean, and its importance ranking with your Stage 1 findings. Note that Optuna picks its best trial by validation accuracy, so confirm it on multiple seeds before claiming it beats the manual result.